# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

# Filter out documents with poor content quality and structure them properly
filtered_docs = []
for doc in loan_complaint_data:
    narrative = doc.metadata.get("Consumer complaint narrative", "")
    
    # Skip documents with insufficient content or too many redactions
    if (len(narrative.strip()) < 100 or 
        narrative.count("XXXX") > 5 or 
        narrative.strip() in ["", "None", "N/A"]):
        continue
    
    # Create meaningful page_content by combining narrative with context
    doc.page_content = f"Customer Issue: {doc.metadata.get('Issue', 'Unknown')}\n"
    doc.page_content += f"Product: {doc.metadata.get('Product', 'Unknown')}\n"
    doc.page_content += f"Complaint Details: {narrative}"
    
    filtered_docs.append(doc)

# Use filtered documents instead
loan_complaint_data = filtered_docs[:20]  # Start with smaller subset


Let's look at an example document to see if everything worked as expected!

In [5]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the provided complaints, include:\n\n1. Errors in loan balances, misapplied payments, and wrongful denials of payment plans.\n2. Incorrect or outdated information on credit reports related to student loans.\n3. Problems with how payments are being handled, such as restrictions on applying extra funds to principal or confusion caused by loan transfers.\n4. Discrepancies in loan account status, including being reported as delinquent or in default without proper basis.\n5. Unfair or confusing interest rate increases, often due to multiple loan sales or changes in servicers.\n6. Receiving bad or misleading information about loan obligations, repayment options, or loan transfer details.\n7. Mishandling of loan data and privacy violations.\n\nWhile these issues vary, a recurring theme is the difficulty borrowers face with loan balance accuracy, payment application, and information transparency. \n\nIn summary, errors and mismanagement related to l

In [12]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, there are cases where the responses from the companies were delayed or where consumers reported waiting over the expected time frames:\n\n- One complaint (row 441, from 03/28/25) was marked as **"Timely response?": "No"**, indicating it was not responded to in a timely manner.\n- Other complaints, such as row 716 (from 05/02/25), received responses marked as **"Timely response?": "Yes"**, suggesting they were handled on time.\n\nTherefore, at least one complaint was explicitly noted as not being handled in a timely manner.'

In [13]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons, including:\n\n1. **Interest Accumulation During Forbearance or Deferment:** Many borrowers reported that while they were able to defer or put loans into forbearance, interest continued to accrue, increasing the total amount owed and making it more difficult to pay off the loans later.\n\n2. **Inability to Afford Increased Payments:** When attempting to increase monthly payments to pay down loans faster, borrowers found these payments unaffordable due to their daily expenses like bills, food, and transportation.\n\n3. **Lack of Clear Communication and Information:** Several complaints highlighted that borrowers were not adequately notified about when their repayment would resume, loan transfers without proper notification, or the specifics of interest and repayment schedules, leading to confusion and missed payments.\n\n4. **Problems with Loan Servicing and Administrative Errors:** Complaints pointed out issues such as loans be

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers. Specific sub-issues include disputes over fees charged, difficulty with payment application (such as paying towards interest instead of principal), and receiving inaccurate or bad information about loan balances and terms. Many complaints mention the lenders or servicers not providing clear explanations, misapplying payments, or failing to respond appropriately.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all the complaints listed include responses labeled as "Timely response?" marked as "Yes." Therefore, it appears that any complaints that were handled were addressed in a timely manner.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n- Problems with managing or understanding their payment plans, such as being steered into the wrong types of forbearances or having automatic payments unenrolled without their knowledge.\n- Lack of proper communication from the loan servicers, leading borrowers to be unaware of their account status, payment requirements, or important updates.\n- Errors or issues with the payment processing system, causing payments to be reversed or not processed correctly.\n- Discrepancies in account information, such as incorrect bank details, which hinder payment processing.\n- Administrative mistakes, like transferring loans between companies without proper notification, which resulted in missed payments and negative impacts on credit scores.\n- Inability to resolve issues promptly due to poor customer service, leading to unresolved problems and accumulating interest or bills.\n\nIn summary, failures to pay back loans often st

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
####✅ Answer: What was the weather in Charlotte yesterday? Because BM25 works better when the query need retrival based on precise keyword matches  vs abstract and semantic generalization needs

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [22]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [23]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, including errors in loan balances, misapplied payments, wrongful denials of payment plans, lack of clear information, and ongoing accumulation of interest despite payments. Many complaints mention issues such as incorrect account information, failure to receive proper documentation, mismanagement during transfers, and difficulties in understanding or managing accrued interest.\n\nTherefore, the most common issue with loans, particularly student loans in this context, seems to be **mismanagement and communication problems related to loan balances, payments, and interest accumulation**.'

In [25]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, some complaints indicate delays in handling or resolution. Specifically, one complaint mentions that it has been nearly 18 months with no resolution and the complainant is still awaiting response and resolution despite over a year passing since submission. Another complaint discusses ongoing issues that are unresolved after 2-3 weeks. \n\nTherefore, yes, there were complaints that did not get handled in a timely manner.'

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of Awareness and Information: Some borrowers were not properly informed by their financial aid officers about the necessity of repayment or the details of their loans, leading to confusion about their repayment obligations.\n\n2. Administrative Issues and Lack of Communication: Borrowers experienced unnotified transfers of their loans between servicers, failure to receive notifications about payments or billing, and difficulty accessing account information online, which hindered their ability to stay current.\n\n3. Accumulation of Interest and Unmanageable Payment Plans: Borrowers faced challenges with interest accumulating on their loans, especially during forbearance or deferment periods, which increased the total amount owed and prolonged repayment. Many found that lowering payments extended the payoff period and increased total interest, making it difficult to pay off the loans.\n\n4. Economic Hardshi

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [27]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [28]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [102]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans include:\n\n- Errors and inaccuracies in loan balances, interest calculations, and account status reports.\n- Problems with loan servicers providing bad information, misapplying payments, or mishandling loan details.\n- Difficulties with payment management, such as inability to apply extra payments toward principal, or payments being applied improperly.\n- Unnotified transfers of loan servicing and updates that impact credit reports and payment obligations.\n- Discrepancies in reported loan status on credit reports, such as incorrect delinquencies or missing payment history.\n- Issues related to loan forgiveness, cancellation, or discharge due to mismanagement or legal disputes.\n- Unauthorized sharing of personal information or improper use of borrower data.\n- Aggressive collection actions and improper communication tactics.\n\nOverall, a recurring theme is that borrowers frequently face inaccuracies, miscommunications

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints and responses, it appears that several complaints were not handled in a timely manner. Specifically, some complaints, such as the one from MOHELA received on 04/18/25 and the follow-up from Maximus Federal Services on 04/21/25, were marked as "timely response? : Yes" but the complaints themselves detail delays, lack of responses, or unresolved issues that persisted over extended periods (e.g., over a year or months). Additionally, complaints like the one from EdFinancial Services logged on 04/04/25 were marked as "timely response? : Yes" but included ongoing issues with no resolution, indicating the complaints were not effectively addressed in a prompt manner.\n\nTherefore, yes, there were complaints that did not get handled in a timely manner, despite some responses being marked as timely, because the issues remained unresolved for long periods or were not adequately addressed.'

In [30]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans due to various systemic issues and mismanagement by loan servicers and lenders. The main reasons include:\n\n1. Inadequate and misleading communication: Borrowers were often not properly informed about their payment obligations, delinquency statuses, or available repayment options, leading to unintentional defaults and negative credit reports.\n\n2. Improper handling of payment plans and interest: Many borrowers experienced late reporting, misapplied payments, or was placed into long-term forbearances without proper guidance. Interest was frequently capitalized without their knowledge, causing balances to balloon and making repayment more difficult.\n\n3. Lack of awareness of legal and alternative repayment options: Borrowers were sometimes steered into forbearances or consolidation without being informed about income-driven repayment plans, loan rehabilitation, or forgiveness programs that could reduce their debts or prevent interest accumulation

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
###✅ Answer: I can't expect the user to be trained on prompting, and my app should handle queries written in all type of prose and loaded with errors. query reformulation would allow the retriver to retrive relevant information that semantically matches several reforumulations of the original query (ideally matching the actul intent of the query better). It increases retrival diversity, by retriving context that may come from lexically variant sources.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [31]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [32]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [33]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [34]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [35]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided data, appears to involve problems with the handling and reporting of the loans. Specifically, issues such as incorrect information on credit reports, errors in loan balances, misapplied payments, wrongful denials of payment plans, and disputes over interest rates are prevalent. Additionally, systemic problems like misreporting, unverified debts, and unfair or deceptive practices are frequently reported by borrowers.\n\nIn summary, problems related to **incorrect or misleading information, mismanagement, and disputes regarding loan balances, interest rates, and reporting** seem to be the most common issues with loans in this context.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, the complaints about the loan applications managed by MOHELA (rows 441 and 84) indicate that the company did not respond within the expected timeframe ("Timely response?": "No"). Similarly, the complaint regarding the dispute settlement with Nelnet (row 474) was handled in a timely manner ("Timely response?": "Yes"), but the consumer reports that they have not yet received a response over 30 days after sending the dispute, which suggests a delay.\n\nOverall, the complaints associated with MOHELA explicitly show they were not handled promptly, indicating that some complaints did not receive timely responses.'

In [38]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors such as financial hardship, mismanagement by educational institutions, lack of proper information about repayment obligations, and issues with loan servicing. For example, some borrowers experienced severe financial hardship after graduation and relied on deferment or forbearance, which increased the total debt due to accumulated interest. Others were misled by schools about the value of their degrees and the long-term financial consequences of taking out loans, making repayment more difficult. Additionally, issues with loan servicing agencies, such as improper notification of payment obligations or problems verifying the legitimacy of debts, also contributed to difficulties in repayment.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [39]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [40]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be problems related to improper handling and mismanagement by loan servicers. This includes errors in loan balances, misapplied payments, wrongful denials of repayment plans, incorrect reporting to credit bureaus, lack of communication or notification regarding account status, and issues with loan transfers and documentation. Many complaints highlight errors in loan information, delays or failures in investigating issues, unfair or aggressive collection practices, and failure to provide proper account access or transparency. \n\nIn summary, the most common issue is *mismanagement and mishandling of student loans by servicers*, leading to financial harm, credit report errors, and communication breakdowns.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, there have been complaints that did not get handled in a timely manner. For example, one complaint (Complaint ID: 12935889) was marked as "No" for timely response, indicating it was not addressed promptly. Additionally, other complaints, such as Complaint ID: 12954720 and 13126709, also show delays or insufficient responses, with some complaints being closed with explanations after significant waiting periods.'

In [43]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans often because of issues such as lack of proper communication, mismanagement, or misleading information from loan servicers. For example, some borrowers were not notified about when their payments were due, their accounts were transferred to new servicers without proper notice, or they were steered into forbearance or deferment options that caused their interest to accumulate significantly. Others faced hardships like job loss, health issues, or financial difficulties, and lacked accessible or supportive repayment options. Additionally, some borrowers encountered errors in their account information, improper reporting to credit bureaus, or faced challenges with loan transfer process and data mishandling. All these factors contributed to their inability to successfully repay their loans.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [44]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [52]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [53]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [54]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [55]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [56]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided complaints data, the most common issues with loans appear to be related to:\n\n- Difficulties and delays in repayment or disputes over repayment plans (e.g., problems with payment plans, incorrect payment amounts, issues with auto-debit setup).\n- Problems with loan servicing, including lack of communication, transparency, and accountability from loan servicers.\n- Issues with credit reporting, such as incorrect defaults, delinquent accounts, or illegal reporting.\n- Mishandling or unauthorized access of personal and financial data, leading to violations of privacy laws.\n- Challenges in obtaining clear information about loan status, balances, or changes in servicer.\n\nOverall, the most recurring issue seems to be **problems with loan servicing and repayment management**, including errors, miscommunications, and delays that affect borrowers’ ability to correctly manage and understand their loans. \n\nIf you have a specific aspect you'd like to focus on, please l

In [57]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints were not handled in a timely manner. Specifically, multiple complaints indicate responses were "Closed with explanation" and involve issues like errors, unresolved disputes, or misconduct, which suggest delays or inadequate handling. \n\nHowever, the detailed information explicitly confirms that at least one complaint was responded to "timely," but given the nature and content of other complaints, it appears that not all complaints received prompt or satisfactory resolution.'

In [58]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including difficulties dealing with their lenders or servicers, administrative or reporting issues, and disputes over the legitimacy or accuracy of the debt. Specific causes mentioned in the complaints include:\n\n- Receiving bad or unclear information about loan status or repayment terms.\n- Problems with loan forgiveness documentation and delays intended to stall or discourage borrowers.\n- Errors or misunderstandings regarding the status of their accounts, such as being reported in default without cause.\n- Payment processing failures or mismatches, leading to missed payments and subsequent default.\n- Disputes over the legitimacy of the debt, especially when borrowers believe their loans are no longer valid or have been improperly reported or transferred.\n- External issues such as legal breaches, privacy violations, or illegal reporting that hinder proper repayment.\n\nIn summary, failures to pay back loans often stem fro

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
### ✅✅✅ Answer: If questions are repetitive, semantic chunking will merge the questions into similar chunks, losing distinction of questions, and providing redundant, blended, generic answers. One way is to not use semantic chunking for questions in FAQ, but alternatively, reduce the chunk size with clear seperators to presever the units within each FAQ. Second option is to treat each questions as a separate chunk. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [73]:
#NLTK Import To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/chrag/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/chrag/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
### Step 0: Dependencies & imports

# included in pyproject.toml file
# "numpy>=2.2.2",
# "ragas==0.3.0",
# "rapidfuzz"
# "langchain-core",
# "langchain-community",
# "langsmith",
#  "tqdm", 


#Setting up the LLM and embedding model and generator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


# Use a more capable model for complex knowledge extraction tasks
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",  # More capable than nano for reasoning tasks
    temperature=0.1,      # Lower temperature for more consistent outputs
    request_timeout=120   # Longer timeout for complex operations
))

generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [ ]:
#Step 1 Create a "golden dataset" a.k.a synthetic test data
# This will generate our knowledge graph under the hood and generate our personas and scenarios to construct our queries

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
sample_docs = loan_complaint_data[:5]  # Try 5 instead of full dataset

try:
    print(f"Generating test dataset with {len(sample_docs)} documents...")
    dataset = generator.generate_with_langchain_docs(
        sample_docs,
        testset_size=2,
    )
    print("Dataset generation completed successfully!")
    print(f"Generated {len(dataset)} test samples")
except Exception as e:
    print(f"Error during dataset generation: {e}")
    print("Try reducing document count or testset_size further")

Applying SummaryExtractor:   0%|          | 0/3 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node d1a37f97-d1c3-4f43-a289-4f86a5bfc4a0 does not have a summary. Skipping filtering.
Node f8b7e1a7-4dcd-44ea-b083-42c9f7cb78eb does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/13 [00:00<?, ?it/s]

unable to apply transformation: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-3.5-turbo in organization org-m71YBvwb05p6pFMRwdurLpCp on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
unable to apply transformation: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-3.5-turbo in organization org-m71YBvwb05p6pFMRwdurLpCp on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}
unable to apply transformation: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-3.5-turbo in organization org-m71YBvwb05p6pFMRwdurLpCp on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

unable to apply transformation: Node 30e09235-a194-4331-9bbc-182dae312fa7 or 807b2c36-10b2-45b8-91b4-97dd116fffde has no entities


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

In [62]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,When did the COVID-19 forbearance program end?,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,Aidvantage why my payment wrong and how they c...,[I submitted my annual Income-Driven Repayment...,Aidvantage assigned me a payment amount that i...,single_hop_specifc_query_synthesizer
2,How does FERPA relate to my student loan debt ...,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,Whaat is the deal with Nelnet and my student l...,"[According to Studentaid.gov, Im to get an ema...","According to the context, the person is confus...",single_hop_specifc_query_synthesizer
4,What does 15 U S C 16811 say about how credit ...,[I am writing to formally dispute inaccurate i...,15 U.S.C. 16811 requires credit reporting agen...,single_hop_specifc_query_synthesizer
5,How can I get Aid Avantage to remove the past ...,[I am devastated. I would like to report a sit...,The individual is seeking to have Aid Avantage...,single_hop_specifc_query_synthesizer
6,Did the Department of Education get my info?,"[On XXXX XXXX XXXX, XXXX XXXX instructed his t...","On XXXX XXXX XXXX, XXXX XXXX instructed his te...",single_hop_specifc_query_synthesizer
7,Why EdFinancials not accept my docs?,[I have provided documentation relating to my ...,The documentation relates to a $5000.00 teache...,single_hop_specifc_query_synthesizer
8,How does FERPA relate to my personal data bein...,[My personal and financial data was compromise...,The context states that personal and financial...,single_hop_specifc_query_synthesizer
9,How does the Privacy Act of 1974 relate to my ...,[I am writing to formally dispute my XXXX XXXX...,The Privacy Act of 1974 is a federal law that ...,single_hop_specifc_query_synthesizer


In [151]:
#all imports
import copy
import time
import pandas as pd
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from langchain_core.runnables import Runnable

In [ ]:
#setting up LangSmith api and project   
import os
import getpass
from langsmith import Client
from langsmith.utils import traceable

os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("🔐 LangSmith API Key: ") # Securely input your LangSmith API key
os.environ["LANGCHAIN_PROJECT"] = "09_Advanced_Retrieval" # Set project name (must be via environment var)
os.environ["LANGCHAIN_TRACING_V2"] = "true" # enables LangSmith's latest tracing system, known as "Tracing V2"

client = Client() # Initialize LangSmith client

In [ ]:
#defines evaluate function for retriever metrics

@traceable(run_type="evaluation", name="Evaluate Retriever", tags=["retriever"])
def evaluate_retriever(
    dataset,
    retriever_chain: Runnable,
    delay_sec: float = 1.0,
    model_name: str = "gpt-4.1-mini",
    timeout: int = 360,
    verbose: bool = False,
    return_dataset: bool = False,
):
    eval_dataset = copy.deepcopy(dataset)

    for i, test_row in enumerate(eval_dataset):
        user_question = test_row.eval_sample.user_input

        if verbose:
            print(f"[{i+1}/{len(eval_dataset)}] Querying retriever: {user_question}")

        result = retriever_chain.invoke({"question": user_question})

        test_row.eval_sample.retrieved_contexts = [
            doc.page_content for doc in result["context"]
        ]
        test_row.eval_sample.response = " "

        time.sleep(delay_sec)

    df = pd.DataFrame([row.eval_sample.to_dict() for row in eval_dataset])
    df["response"] = df["response"].fillna(" ")
    ragas_dataset = EvaluationDataset.from_pandas(df)

    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model=model_name))
    run_config = RunConfig(timeout=timeout)

    retriever_metrics = [
        LLMContextRecall(),        # Measures how much of the relevant context (needed to answer the query) is retrieved
        ContextEntityRecall(),     #  Measures whether key entities from the gold/reference answer are present in the retrieved context.
        ContextPrecision(),        #  easures the proportion of relevant chunks in the retrieved contexts
        NoiseSensitivity(),         #  easures how often a system makes errors by providing incorrect responses
    ]

    results = evaluate(
        dataset=ragas_dataset,
        metrics=retriever_metrics,
        llm=evaluator_llm,
        run_config=run_config,
    )

    return (results, eval_dataset) if return_dataset else results

In [153]:
 # Copy dataset to avoid modifying original
eval_dataset = copy.deepcopy(dataset)

In [ ]:
# Call the evaluator function for naive retriever
naive_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=naive_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(naive_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chrag/projects/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice

{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


### {'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for bm25
bm25_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=bm25_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(bm25_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[37]: TimeoutError()


{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


### {'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for Cohere Reranker
CohereReranker_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=contextual_compression_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(CohereReranker_results)




In [ ]:
# Call the evaluator function  for MultiQueryRetriever
MultiQueryRetriever_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=multi_query_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(MultiQueryRetriever_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[15]: TimeoutError()
Exception raised in Job[25]: TimeoutError()


{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


###{'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for Parent Document Retriever
parent_document_retriever_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=parent_document_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(parent_document_retriever_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


###{'context_recall': 0.9500, 'context_entity_recall': 0.5708}

In [ ]:
# Call the evaluator function  for Ensemble Retriever
ensemble_retrieval_chain_results = evaluate_retriever(
    dataset=eval_dataset,
    retriever_chain=ensemble_retrieval_chain,  # 👈 can be swapped with other retrievers
    verbose=True
)

print(ensemble_retrieval_chain_results)

[1/20] Querying retriever: When did the COVID-19 forbearance program end?
[2/20] Querying retriever: Aidvantage why my payment wrong and how they calc it?
[3/20] Querying retriever: How does FERPA relate to my student loan debt cancellation request?
[4/20] Querying retriever: Whaat is the deal with Nelnet and my student loan issuse?
[5/20] Querying retriever: What does 15 U S C 16811 say about how credit bureaus have to do reinvestigation and fix wrong info?
[6/20] Querying retriever: How can I get Aid Avantage to remove the past due notices from my accounts?
[7/20] Querying retriever: Did the Department of Education get my info?
[8/20] Querying retriever: Why EdFinancials not accept my docs?
[9/20] Querying retriever: How does FERPA relate to my personal data being compromised and my request for student loan debt cancellation?
[10/20] Querying retriever: How does the Privacy Act of 1974 relate to my issues with student loan information disclosure?
[11/20] Querying retriever: What is H

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[9]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[3]: TimeoutError()


{'context_recall': 0.9500, 'context_entity_recall': 0.5708}


###{'context_recall': 0.9500, 'context_entity_recall': 0.5708}